<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_logitrack_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *LogiTrack — Supply Chain Analytics* dans Power BI Desktop. C'est un dashboard d'aide au pilotage logistique pour 30 corridors en Afrique de l'Ouest.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| 🎓 | Une méthode opérationnelle pour construire un visuel précis |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |

### Le contexte métier

**LogiTrack** est un transporteur multimodal qui opère sur 30 corridors entre 9 pays d'Afrique de l'Ouest (Côte d'Ivoire, Sénégal, Ghana, Mali, Burkina Faso, Cameroun, etc.). Le dashboard pilote 4 enjeux :

1. **SLA Breach** — taux de livraisons hors délai contractuel (cible interne < 35 %)
2. **CSAT** — satisfaction client (cible ≥ 4.0/5, alerte < 3.5)
3. **Pénalités financières** — coût direct des breaches (FCFA)
4. **Risque ML** — modèle de scoring qui prédit les retards à venir (alertes critiques score > 0.75)

Le dashboard répond à 5 questions :

| Page | Question |
|---|---|
| 1 — Vue Executive | Quelle est la santé globale du réseau logistique ? |
| 2 — Corridors | Quels corridors sont à risque (long, breach élevé, pénalités) ? |
| 3 — Transporteurs | Quels transporteurs sous-performent et lesquels sont nos références ? |
| 4 — CSAT & Qualité | La satisfaction client est-elle corrélée au taux de breach ? |
| 5 — Alertes ML | Quelles livraisons en cours nécessitent une intervention immédiate ? |

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — sources brutes vs outputs ML/EDA

Le projet utilise **deux familles de sources** :

**Sources brutes** (3 tables) :
- `livraisons` — 15 000 livraisons avec 33 colonnes (origine, destination, dates, SLA, retards, breach, escalade, CSAT, pénalité)
- `transporteurs` — 14 transporteurs actifs avec leur tier de fiabilité
- `Calendrier` — table de dates générée

**Outputs Python** (7 tables agrégées) :
- `logitrack_analytics` — vue préparée des livraisons (56 cols enrichies)
- `logitrack_corridors` — 1 ligne = 1 corridor avec distance, risque douanier, breach %
- `logitrack_transporteurs_perf` — performance par transporteur (tier_fiabilite, retards, breaches)
- `logitrack_mensuel` — KPIs agrégés par mois
- `logitrack_cout_retard` — coût estimé des retards par corridor
- `logitrack_entrepots_perf` — performance entrepôts d'origine
- `logitrack_risque_scores` — sortie ML : 1 ligne = 1 livraison avec score de risque + niveau urgence

### ⚠️ Piège fréquent — quelle table pour quel KPI ?

- **KPIs volume / breach / CSAT** ⇒ `livraisons` (table de référence)
- **Top 5 corridors / risque douanier** ⇒ `logitrack_corridors` (LOOKUPVALUE)
- **Tier fiabilité transporteur** ⇒ `logitrack_transporteurs_perf`
- **Évolution mensuelle** ⇒ `logitrack_mensuel`
- **Top 5 alertes prioritaires** ⇒ `logitrack_risque_scores` (sortie ML NB5)

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit.

### Les 10 URLs à utiliser

```
# Sources brutes
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/data/livraisons.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/data/transporteurs.csv

# Outputs Python (sortie notebooks 1-5)
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_analytics.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_corridors.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_transporteurs_perf.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_mensuel.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_cout_retard.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_entrepots_perf.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/corrige/outputs/logitrack_risque_scores.csv
```





<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/01.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

Sans ça, Power BI crée une LocalDateTable cachée pour chaque colonne de date — sur ce projet (`reservations[date_arrivee]`, `reservations[date_depart]`, `paiements[date_paiement]`, `services[date_service]`...), c'est 4-5 tables fantômes en moins.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>


### ⚠️ Piège — la colonne `csat`

La colonne `livraisons[csat]` est typée **String** avec séparateur décimal anglo-saxon (`4.5` au lieu de `4,5`). Avant tout calcul, applique : `VALUE(SUBSTITUTE([csat], ".", ","))`. Sinon `AVERAGE` plante en locale FR.

---
# II — Modéliser les données

## 2.1 Schéma en étoile

### 📘 Concept clé — `livraisons` est la table pivot, les outputs Python sont des satellites

```
                   +------------------+
                   |   Calendrier     |
                   +--------+---------+
                            | 1
                            | N
    +----------------+   1   +-----------+   N   +-----------------+
    | transporteurs  +-------+ livraisons +-------+ logitrack_      |
    +----------------+       +------+----+       | risque_scores   |
                                    | 1                +-----------------+
                                    |
                       (clés natifs : corridor, transporteur_id, date)

Tables agrégées (autonomes ou liées par lookup) :
  - logitrack_corridors          ← LOOKUPVALUE sur livraisons[Corridor]
  - logitrack_transporteurs_perf ← rel 1-1 sur transporteurs[nom] = perf[transporteur]
  - logitrack_mensuel            ← rel sur Calendrier[Date]
  - logitrack_cout_retard        ← rel sur livraisons[Corridor]
  - logitrack_entrepots_perf     ← rel sur livraisons[entrepot_origine]
  - logitrack_analytics          ← vue préparée, peut remplacer livraisons selon mesures
```

## 2.2 Créer la table Calendrier

**Modélisation → Nouvelle table** :

```dax
Calendrier = 
ADDCOLUMNS(
    CALENDAR(DATE(2023,1,1), DATE(2024,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR")
)
```

## 2.3 Marquer Calendrier comme table de dates

Vue Données → `Calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/04.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 2.4 Établir les relations

| # | De (1) | Clé | Vers (N) | Clé | Direction |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `livraisons` | `date_creation` | Single |
| 2 | `Calendrier` | `Date` | `logitrack_mensuel` | `mois` | Single |
| 3 | `transporteurs` | `transporteur_id` | `livraisons` | `transporteur_id` | Single |
| 4 | `transporteurs` | `nom` | `logitrack_transporteurs_perf` | `transporteur` | Single |
| 5 | `livraisons` | `livraison_id` | `logitrack_risque_scores` | `livraison_id` | Single |

### ⚠️ Les seules relations actives à conserver

Power BI peut détecter automatiquement d'autres relations entre les outputs Python (`logitrack_corridors` ↔ `logitrack_cout_retard` sur `corridor`, `logitrack_corridors` ↔ `logitrack_entrepots_perf` sur `rang_risque`, etc.). **Désactive-les toutes** : ces tables agrégées sont accédées via `LOOKUPVALUE` dans les mesures, jamais par relation. Garder uniquement ces 5 relations évite les filtres croisés indésirables et conserve un schéma en étoile propre.

**Comment vérifier** : *Modélisation → Gérer les relations* → seules les 5 lignes ci-dessus doivent être marquées **Active**. Les autres : décocher *Activer cette relation* ou supprimer.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/02.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>


---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive.

---
# IV — Construire les 56 mesures DAX

### Vue d'ensemble des 8 dossiers

| # | Dossier | Mesures | Rôle |
|---|---|---|---|
| 0 | Pilotage Opérationnel | 5 | Sous-titres dynamiques + bandeau alerte |
| 1 | KPIs Globaux | 8 | Total Livraisons, Taux Breach %, CSAT, Pénalités, Escalades |
| 2 | Évolution | 3 | Variation Breach %, YTD |
| 3 | Alertes ML | 4 | Nb Alertes Critiques, Bandeau, Quadrant Transporteur |
| 4 | Vue Corridors | 6 | Distance, Risque Douanier, Retard Moyen, Rang, Pénalités K FCFA |
| 5 | Vue Transporteurs | 9 | Nb Actifs/Conformes/À Surveiller, Variation Retard, Rang |
| 6 | Vue CSAT | 7 | CSAT Moyen, Pct CSAT≥4, Variation, Total Escalades |
| 7 | Vue Alertes ML | 7 | Nb Alertes Élevées, Score Risque Moyen, Niveau Urgence Icon, Rang |
| _ | _Helpers | 3 | Mesures couleur dynamiques |

## 4.1 Dossier `1. KPIs Globaux` (8 mesures)

```dax
Total Livraisons = COUNTROWS(livraisons)

Taux Breach % = 
DIVIDE(
    CALCULATE(COUNTROWS(livraisons), livraisons[sla_breach] = 1),
    COUNTROWS(livraisons)
) * 100

Taux SLA Breach Ratio = 
DIVIDE(
    CALCULATE(COUNTROWS(livraisons), livraisons[sla_breach] = 1),
    COUNTROWS(livraisons)
)

Taux Conforme Ratio = 1 - [Taux SLA Breach Ratio]

Nb Livraisons Breach = CALCULATE(COUNTROWS(livraisons), livraisons[sla_breach] = 1)

CSAT Moyen = 
AVERAGEX(
    FILTER(livraisons, NOT(ISBLANK(livraisons[csat]))),
    IFERROR(VALUE(SUBSTITUTE(livraisons[csat], ".", ",")), BLANK())
)

Total Penalites FCFA = SUM(livraisons[penalite_fcfa])

Nb Escalades = CALCULATE(COUNTROWS(livraisons), livraisons[escalade] = 1)
```

### ⚠️ Deux versions du Taux Breach : `%` vs `Ratio`

- `Taux Breach %` retourne **38.5** (déjà × 100, format `0.0\"%\"`) — pour cards et bandeau
- `Taux SLA Breach Ratio` retourne **0.385** (ratio 0-1, format Pourcentage) — pour barres empilées 100 %

Cette dualité existe car certains visuels Power BI attendent un ratio (les barres empilées 100 %), d'autres préfèrent un nombre déjà formaté.

## 4.2 Dossier `2. Évolution` (3 mesures)

```dax
Taux Breach Mois Précédent % = 
CALCULATE([Taux Breach %], DATEADD(Calendrier[Date], -1, MONTH))

Variation Breach % = 
VAR _curr = [Taux Breach %]
VAR _prev = [Taux Breach Mois Précédent %]
RETURN IF(ISBLANK(_prev), BLANK(), _curr - _prev)

Taux Breach YTD % = 
CALCULATE([Taux Breach %], DATESYTD(Calendrier[Date]))
```

*La variation est en **points de pourcentage** (pas en %) : on soustrait deux taux, on ne fait pas un DIVIDE. Une variation de +2 pp = le breach est passé de 38 % à 40 %.*

## 4.3 Dossier `3. Alertes ML` (4 mesures)

```dax
Nb Alertes Critiques = 
CALCULATE(
    COUNTROWS(logitrack_risque_scores),
    logitrack_risque_scores[score_risque] > 0.75
)

Nb Alertes Total = COUNTROWS(logitrack_risque_scores)

Bandeau Alerte = 
VAR _n = [Nb Alertes Critiques]
RETURN IF(
    _n > 0,
    "⚠ " & _n & " livraisons en risque critique — score > 0.75 · Action immediate requise",
    "✓ Aucune alerte critique en cours · Niveau de service nominal"
)

Quadrant Transporteur = 
VAR _breach = [Taux Breach %]
VAR _csat = [CSAT Moyen]
RETURN SWITCH(TRUE(),
    _breach < 35 && _csat >= 4, "Top Performer",
    _breach < 35 && _csat <  4, "Fiable Peu Apprécié",
    _breach >= 35 && _csat >= 4, "Apprécié Peu Fiable",
    "A Surveiller"
)
```

## 4.4 Dossier `4. Vue Corridors` (6 mesures)

```dax
Distance Corridor = 
LOOKUPVALUE(
    logitrack_corridors[distance_km],
    logitrack_corridors[corridor], MAX(livraisons[Corridor])
)

Risque Douanier = 
LOOKUPVALUE(
    logitrack_corridors[risque_douanier],
    logitrack_corridors[corridor], MAX(livraisons[Corridor])
)

Risque Douanier Icon = 
SWITCH([Risque Douanier],
    "Eleve",  "🔴 Eleve",
    "Moyen",  "🟠 Moyen",
    "Faible", "🟢 Faible",
    ""
)

Retard Moyen Jours Breach = 
CALCULATE(
    AVERAGE(livraisons[retard_jours]),
    livraisons[sla_breach] = 1
)

Rang Risque Breach = 
RANKX(
    ALL(livraisons[Corridor]),
    CALCULATE([Taux Breach %]),
    ,
    DESC,
    DENSE
)

Penalites K FCFA = DIVIDE([Total Penalites FCFA], 1000)
```

### ⚠️ Piège — `RANKX` qui renvoie 1 partout

Sur un visuel table, sans le `CALCULATE([Taux Breach %])` interne, `RANKX` n'arrive pas à transitionner du contexte de ligne au contexte de filtre. Tous les rangs renvoient 1. **Toujours wrapper la mesure dans CALCULATE quand RANKX itère sur ALL.**

## 4.5 Dossier `5. Vue Transporteurs` (9 mesures)

```dax
Nb Transporteurs Actifs = 
CALCULATE(COUNTROWS(transporteurs), transporteurs[actif] = TRUE())

Nb Transporteurs Conformes = 
CALCULATE(
    DISTINCTCOUNT(logitrack_transporteurs_perf[transporteur_id]),
    logitrack_transporteurs_perf[tier_fiabilite] IN {1, 2}
)

Nb Transporteurs A Surveiller = 
CALCULATE(
    DISTINCTCOUNT(logitrack_transporteurs_perf[transporteur_id]),
    logitrack_transporteurs_perf[tier_fiabilite] = 3
)

Sous Titre Transporteurs Actifs = 
[Nb Transporteurs Conformes] & " conformes · " & [Nb Transporteurs A Surveiller] & " à surveiller"

Variation Retard Jours = 
VAR _curr = AVERAGE(livraisons[retard_jours])
VAR _prev = CALCULATE(
    AVERAGE(livraisons[retard_jours]),
    DATEADD(Calendrier[Date], -1, QUARTER)
)
RETURN IF(ISBLANK(_prev), BLANK(), _curr - _prev)

Sous Titre Retard Moyen = 
VAR _v = [Variation Retard Jours]
RETURN IF(ISBLANK(_v), "",
    IF(_v > 0, "↑ +" & FORMAT(_v, "0.0"), "↓ " & FORMAT(_v, "0.0"))
)

Penalites par Transporteur K FCFA = 
DIVIDE([Total Penalites FCFA], [Nb Transporteurs Actifs] * 1000)

Sous Titre Penalites par Transporteur = 
"Moyenne sur " & [Nb Transporteurs Actifs] & " transporteurs"

Rang Transporteur Breach = 
RANKX(
    ALL(transporteurs[nom]),
    CALCULATE([Nb Livraisons Breach]),
    ,
    DESC,
    DENSE
)
```

### 📘 Sémantique DOWN-IS-GOOD pour la variation retard

Pour la plupart des KPIs, ↑ = bon. Pour le retard moyen, **↓ = bon** (un retard qui baisse est une amélioration). La mesure `Color Variation Retard` du dossier `_Helpers` retourne vert si delta ≤ 0, rouge sinon.

## 4.6 Dossiers `6-7-_Helpers` — CSAT + Alertes ML + Couleurs (17 mesures)

### `6. Vue CSAT` (7 mesures)

```dax
CSAT Moyen Global = [CSAT Moyen]  // alias avec sémantique page-CSAT

Pct Livraisons CSAT Sup 4 = 
DIVIDE(
    CALCULATE(
        COUNTROWS(livraisons),
        IFERROR(VALUE(SUBSTITUTE(livraisons[csat], ".", ",")), 0) >= 4
    ),
    CALCULATE(COUNTROWS(livraisons), NOT(ISBLANK(livraisons[csat])))
)

Variation CSAT Sup 4 = 
VAR _curr = [Pct Livraisons CSAT Sup 4]
VAR _prev = CALCULATE([Pct Livraisons CSAT Sup 4], DATEADD(Calendrier[Date], -1, QUARTER))
RETURN IF(ISBLANK(_prev), BLANK(), (_curr - _prev) * 100)

Total Escalades = CALCULATE(COUNTROWS(livraisons), livraisons[escalade] = 1)
Pct Escalades = DIVIDE([Total Escalades], [Total Livraisons])
```

### `7. Vue Alertes ML` (7 mesures)

```dax
Nb Alertes Élevées = 
CALCULATE(
    COUNTROWS(logitrack_risque_scores),
    logitrack_risque_scores[score_risque] >= 0.55,
    logitrack_risque_scores[score_risque] <= 0.75
)

Nb Alertes Moyens = 
CALCULATE(
    COUNTROWS(logitrack_risque_scores),
    logitrack_risque_scores[score_risque] >= 0.30,
    logitrack_risque_scores[score_risque] < 0.55
)

Score Risque Moyen = AVERAGE(logitrack_risque_scores[score_risque])

Niveau Urgence Icon = 
SWITCH(SELECTEDVALUE(logitrack_risque_scores[niveau_urgence]),
    "Critique", "🔴",
    "Eleve",    "🟠",
    "Modere",   "🟢",
    ""
)

Rang Score Risque = 
IF(HASONEVALUE(logitrack_risque_scores[livraison_id]),
    RANKX(
        ALL(logitrack_risque_scores[livraison_id]),
        CALCULATE(MAX(logitrack_risque_scores[score_risque])),
        ,
        DESC,
        DENSE
    )
)
```

### `_Helpers` (3 mesures couleur)

```dax
Color Variation Retard = 
// DOWN IS GOOD : vert si retard a diminué
VAR _v = [Variation Retard Jours]
RETURN IF(_v <= 0, "#10D9A3", "#FF4D6D")

Color Variation CSAT = 
// UP IS GOOD : vert si CSAT a augmenté
VAR _v = [Variation CSAT Sup 4]
RETURN IF(_v >= 0, "#10D9A3", "#FF4D6D")

Color Score Risque = 
// 3 paliers selon niveau urgence
SWITCH(SELECTEDVALUE(logitrack_risque_scores[niveau_urgence]),
    "Critique", "#FF4D6D",
    "Eleve",    "#FFB547",
    "Modere",   "#10D9A3",
    "#7891B5"
)
```

---
# V — Design system

## 5.1 Charte LogiTrack — Dark + Cyan (A télécharger sur le plateforme)

| Rôle | Hex | Usage |
|---|---|---|
| Fond page | `#0B1421` ou `#0F1A2C` | Arrière-plan général |
| Card background | `#1A2436` | Fond des cards |
| Primaire (cyan) | `#00D4D4` | Logo, titre actif sidebar, KPI Total Livraisons |
| Vert mint | `#10D9A3` | Conforme, variation positive (selon UP/DOWN logic), Top Performer |
| Rouge corail | `#FF4D6D` | SLA Breach, alertes critiques, dégradation |
| Orange | `#FFB547` | Alertes élevées, pénalités, intermédiaire |
| Texte principal | `#FFFFFF` | Hero numbers, titres |
| Texte secondaire | `#B0B8C7` | Labels, axes, sous-titres |
| Bordure subtile | `#2A3548` | Cards, séparateurs |

### Typographie

| Élément | Police | Taille | Poids |
|---|---|---|---|
| Titre de page | Georgia | 44 | 700 |
| Sous-titre | Segoe UI | 14 | 300 |
| Hero KPI | Segoe UI | 36 | 700 |
| Label KPI | Segoe UI | 13 | 400 (gris clair) |
| Navbar item | Segoe UI | 14 | 500 |

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes. Méthode pro : dessiner dans **PowerPoint** (mockup vierge), exporter en **PNG haute résolution** (1280×720), importer comme **arrière-plan de page**, poser les visuels Power BI **par-dessus**.

### Ce que le mockup PPTX doit contenir

✅ Logo LogiTrack (carré cyan avec icône camion blanc) en haut-gauche, navbar horizontale en haut, footer DataProjectLab cyan, cards `#1A2436` avec bordure subtile.

❌ Pas de titre de page, pas de slicers, pas de KPI valeurs, pas de chart data.

### 🔧 Méthode 1 — Export PNG depuis PowerPoint à 150 DPI

1. **Win + R** → `regedit` → `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
2. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)** → Nom : `ExportBitmapResolution`, Valeur : `150`
3. Redémarrer PowerPoint, **Fichier → Enregistrer sous → PNG → Toutes les diapositives**

### 🔧 Méthode 2 — CloudConvert

[cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png) → upload `mockup_logitrack_blank.pptx` → 150 DPI → 1280×720.

### Renommage final

```
bg-01-vue-executive.png
bg-02-corridors.png
bg-03-transporteurs.png
bg-04-csat.png
bg-05-alertes-ml.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page**
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement** → **Adapter** · **Transparence** → **0 %**

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/03.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VI — Construire les 5 pages

Cette partie détaille **chaque visuel** avec sa configuration exacte (type, axes, couleurs, étiquettes) et les **méthodes Power BI** non-triviales nécessaires pour le rendu final.

## 6.1 Page 1 — Vue Executive

> *« Quelle est la santé globale du réseau logistique ? »*

**1. Bandeau d'alerte critique**

- Type : Zone de texte avec valeur dynamique
- Texte : `[Bandeau Alerte]` (ex: « ⚠ 25 livraisons en risque critique — score > 0.75 · Action immediate requise »)
- Fond : `#3D1421` (rouge très foncé) si alerte présente, `#0F2C20` (vert très foncé) sinon
- Bordure gauche : 4px rouge corail `#FF4D6D` ou vert mint `#10D9A3`
- Police : Segoe UI Bold 14pt rouge `#FF4D6D` ou vert

> 🎓 **METHODE — Bandeau dynamique avec couleur conditionnelle**
>
> Power BI ne permet pas de changer la couleur de fond d'une zone de texte par formule. Solution :
>
> 1. Créer 2 zones de texte superposées (une rouge, une verte) avec leur fond respectif
> 2. Sur chacune, configurer **Format → Visibilité** → fx → conditionnelle selon `[Nb Alertes Critiques] > 0`
> 3. Alternative simple : utiliser un visuel **Carte** classique avec `[Bandeau Alerte]`, et appliquer **Couleur de fond** → fx → mesure couleur dédiée

**2-5. 4 KPI cards**

- KPI 1 — **Total Livraisons** : icône 🚚 cyan, valeur `15 000` blanc 36pt
- KPI 2 — **Taux SLA Breach** : icône ⚠️ rouge, valeur `39,0 %` rouge `#FF4D6D` 36pt
- KPI 3 — **CSAT Moyen** : icône ⭐ vert mint, valeur `3,65` vert `#10D9A3` 36pt
- KPI 4 — **Pénalités Totales** : icône 💰 orange, valeur `280 735` orange `#FFB547` 36pt

Format card : fond `#1A2436`, coins arrondis 8px, padding 24px, bordure subtile 1px `#2A3548`.

**6. Volume mensuel de livraisons**

- Type : Barres verticales
- Axe X : `Calendrier[Mois_Nom]`
- Axe Y : `[Total Livraisons]`
- Couleur barres : cyan `#00D4D4`
- Étiquettes : aucune (la lecture des barres suffit)
- Sous-titre : `[Sous Titre Volume Mensuel]` (ex: « Évolution du volume total · 12 mois cumulés »)

**7. Taux de SLA Breach mensuel**

- Type : Ligne avec marqueurs et étiquettes
- Axe X : `Calendrier[Mois_Nom]`
- Axe Y : `[Taux Breach %]`
- Couleur ligne : rouge corail `#FF4D6D`, épaisseur 2px
- Marqueurs : ronds rouge taille 6
- Étiquettes : valeur en % au-dessus de chaque point (40,0 % / 38,8 % / etc.)
- Sous-titre : `[Sous Titre Taux Breach]` (ex: « Tendance — 39,0% des livraisons en retard sur la période »)



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/01_page_vue_executive_.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.2 Page 2 — Corridors

> *« Quels corridors sont à risque (long, breach élevé, pénalités) ? »*

**1. Taux de livraisons SLA breachées par corridor (barres empilées 100 %)**

- Type : **Histogramme groupé empilé 100 %** (barres horizontales)
- Axe Y : `livraisons[Corridor]`
- Axe X (empilé 100 %) : 2 séries
  - Série 1 — `[Taux SLA Breach Ratio]` couleur rouge corail `#FF4D6D` (label « SLA Breach % »)
  - Série 2 — `[Taux Conforme Ratio]` couleur vert mint `#10D9A3` (label « Conforme % »)
- Filtre : Top 5 par `[Taux Breach %]` DESC
- Tri : descendant
- Légende : en bas (« SLA Breach % » rouge / « Conforme % » vert)
- Pas d'étiquettes (la longueur des segments parle d'elle-même)

> 🎓 **METHODE — Barres empilées 100 % avec 2 mesures complémentaires**
>
> Power BI ne fait pas naturellement de barre 100 % avec 2 mesures (il faut une dimension catégorielle). Astuce :
>
> 1. Créer la mesure complémentaire `Taux Conforme Ratio = 1 - [Taux SLA Breach Ratio]`
> 2. Visuel **Histogramme empilé 100 %** : Axe Y = corridor, **Valeurs** = glisser les 2 mesures (Breach Ratio et Conforme Ratio)
> 3. Power BI affiche automatiquement les 2 séries empilées sur 100 %, avec une couleur par mesure

**2. Distance vs Taux breach (scatter)**

- Type : Nuage de points (Scatter chart)
- Détails : `livraisons[Corridor]`
- Axe X : `[Distance Corridor]`
- Axe Y : `[Taux Breach %]`
- Taille des bulles : `[Total Livraisons]` (volume = bulle plus grande)
- Couleur des bulles : vert mint `#10D9A3` uniforme (pour ne pas surcharger visuellement)
- Sous-titre : « Corrélation : corridors longs → plus de breach »

> 🎓 **METHODE — Bulles taille = volume**
>
> Le scatter natif Power BI a un champ **Taille** dédié. Glisser `[Total Livraisons]` dans Taille — Power BI normalise automatiquement les rayons des bulles entre min et max. Si certaines bulles sont trop petites, Format → Marqueurs → Taille minimale = 8px.

**3. Détail des 7 corridors les plus critiques**

- Type : Table
- Filtre : Top 7 par `[Taux Breach %]` DESC
- Colonnes :
  - `livraisons[Corridor]`
  - `[Risque Douanier Icon]` ("🔴 Eleve" / "🟠 Moyen" / "🟢 Faible")
  - `[Total Livraisons]` (label « Livraisons »)
  - `[Distance Corridor]` formaté `0" km"`
  - `[Taux Breach %]` formaté `0,0\" %\"` couleur conditionnelle (rouge si > 70 %)
  - `[Retard Moyen Jours Breach]` formaté `0,0\" j\"`
  - `[Penalites K FCFA]` formaté `0,0\" K\"`
  - `[Rang Risque Breach]` formaté `"#"0`

> 🎓 **METHODE — Couleur conditionnelle Breach % dans la table**
>
> Sur la colonne `[Taux Breach %]` : Format → **Cellules** → **Couleur de la police** → fx → **Mettre en forme par : Règles** → règle « Si valeur ≥ 70 alors `#FF4D6D` ». Alternative pour un dégradé continu : utiliser **Mise en échelle des couleurs** (vert pâle → rouge vif).

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/02_page_corridors_.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.3 Page 3 — Transporteurs

> *« Quels transporteurs sous-performent et lesquels sont nos références ? »*

**1-3. 3 KPI cards Transporteurs**

- KPI 1 — **Transporteurs Actifs** : icône 🚚 cyan, valeur `[Nb Transporteurs Actifs]` (14) blanc 36pt, sous-texte `[Sous Titre Transporteurs Actifs]` (« 10 conformes · 4 à surveiller »)
- KPI 2 — **Retard Moyen** : icône ⏱ orange, valeur `6,3` orange `#FFB547` 36pt, sous-texte `[Sous Titre Retard Moyen]` avec `[Color Variation Retard]` (« ↓ -0,1 » en vert si baisse)
- KPI 3 — **Pénalités / Transporteur** : icône 💰 orange, valeur `[Penalites par Transporteur K FCFA]` formatée `0,0\" K\"` (ex: « 20,1 K »), sous-texte `[Sous Titre Penalites par Transporteur]` (« Moyenne sur 14 transporteurs »)

**4. Top 8 transporteurs par taux de breach**

- Type : Barres horizontales
- Axe Y : `transporteurs[nom]`
- Axe X : `[Taux Breach %]`
- Filtre : Top N = 8 par `[Taux Breach %]` DESC (les pires)
- Couleur barres : rouge corail `#FF4D6D` uniforme
- Étiquettes : valeur en `%` à droite des barres (ex: « 38% », « 39% », ..., « 43% »)
- Tri : descendant

**5. Classement détaillé (table)**

- Type : Table
- Colonnes :
  - `transporteurs[nom]` (label « Transporteur »)
  - `[Total Livraisons]` (label « Livraison »)
  - `[Total Penalites FCFA]` formaté `0,0\" K\"` (label « Penalites »)
- Tri : ascendant alphabétique sur Transporteur (ou personnalisable)
- Ligne **Total** activée (Format → Total → Activer) avec valeurs agrégées



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/03_page_transporteurs_.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.4 Page 4 — CSAT & Qualité Service

> *« La satisfaction client est-elle corrélée au taux de breach ? »*

**1-3. 3 KPI cards CSAT**

- KPI 1 — **CSAT Moyen Global** : icône ⭐ vert mint, valeur `3,65 / 5` blanc 36pt, sous-texte `[Sous Titre CSAT Moyen]` (« Sur 9 560 livraisons »)
- KPI 2 — **% Livraisons CSAT ≥ 4** : icône ✓ cyan, valeur `[Pct Livraisons CSAT Sup 4]` formatée `0,0\"%\"` (41,8%), sous-texte avec `[Color Variation CSAT]` (« +0,3 pts » vert si hausse)
- KPI 3 — **Total Escalades** : icône 🚨 rouge, valeur `[Total Escalades]` (920) rouge `#FF4D6D` 36pt, sous-texte `[Sous Titre Total Escalades]` (« 6,1% du volume »)

**4. Évolution mensuelle CSAT vs Taux Breach (combo chart)**

- Type : **Histogramme et courbe groupés** (combo chart, 2 axes Y)
- Axe X : `Calendrier[Mois_Nom]`
- Axe Y1 (gauche, ligne) : `[CSAT Moyen]` couleur cyan `#00D4D4`
- Axe Y2 (droite, ligne) : `[Taux Breach %]` couleur rouge corail `#FF4D6D`
- Marqueurs : ronds taille 5 sur chaque point
- Légende : en bas (« CSAT Moyen » cyan / « Taux Breach % » rouge)
- Sous-titre : « Corrélation inverse : CSAT baisse quand breach monte »

> 🎓 **METHODE — 2 lignes superposées avec axes Y indépendants**
>
> Le visuel **Ligne et histogramme empilé** ou **Histogramme et courbe groupés** supporte nativement 2 axes Y. Mais ici on veut **2 lignes** (pas 1 barre + 1 ligne). Astuce :
>
> 1. Utiliser le visuel **Graphique en courbes** simple
> 2. Glisser les 2 mesures dans **Valeurs**
> 3. Format → **Axe Y** → Plage de valeurs → activer **Axe secondaire** pour la 2ᵉ mesure
> 4. Personnaliser les couleurs de chaque ligne via Format → Couleurs des données

**5. Corridors avec CSAT le plus bas (table)**

- Type : Table
- Colonnes :
  - `livraisons[Corridor]`
  - `[CSAT Moyen]` formaté `0,00` (label « CSAT ») couleur conditionnelle rouge si < 3,5
  - `[Taux Breach %]` formaté `0,0\" %\"` (label « Breach »)
  - `[Nb Escalades]` (label « Esc. »)
- Tri : ascendant sur CSAT (les pires en haut)
- Filtre : Top N = 10 sur CSAT ascendant
- Couleurs lignes : alternance fond `#1A2436` / `#1F2942`



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/04_page_csat_.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.5 Page 5 — Alertes ML & Prédictions de Retard

> *« Quelles livraisons en cours nécessitent une intervention immédiate ? »*

**1. Bandeau d'alerte critique (haut de page)**

- Type : Zone de texte avec valeur dynamique
- Texte : `[Bandeau Alerte]` (« ⚠ 25 livraisons en risque critique — score > 0.75 · Action immediate requise »)
- Fond : `#3D1421` (rouge très foncé), bordure 2px rouge `#FF4D6D`, coins arrondis 4px
- Police : Segoe UI 14pt rouge corail

**2-4. 3 KPI cards Alertes**

- KPI 1 — **Alertes Critiques** : icône cercle rouge, valeur `[Nb Alertes Critiques]` (25) rouge 36pt, sous-texte `[Sous Titre Alertes Critiques]` (« Score > 0.75 »)
- KPI 2 — **Alertes Élevées** : icône cercle orange, valeur `[Nb Alertes Élevées]` (2) orange 36pt, sous-texte `[Sous Titre Alertes Elevees]` (« Score 0.55 — 0.75 »)
- KPI 3 — **Score Risque Moyen** : icône bar chart cyan, valeur `[Score Risque Moyen]` formaté `0,00` (0,83) cyan 36pt, sous-texte `[Sous Titre Score Risque Moyen]` (« Sur 30 alertes ouvertes »)

**5. Top 5 alertes prioritaires (table)**

- Type : Table avec icône en colonne d'en-tête (👑)
- Filtre : Top N = 5 sur `[Score Risque Moyen]` DESC
- Colonnes :
  - `logitrack_risque_scores[livraison_id]` (label « ID »)
  - `logitrack_risque_scores[origine_pays]` (label « Origine »)
  - `logitrack_risque_scores[destination_pays]` (label « Destination »)
  - `logitrack_risque_scores[priorite]` (label « Priorité »)
  - `logitrack_risque_scores[date_prevue]` formaté date longue (label « Date prév. »)
  - `logitrack_risque_scores[score_risque]` formaté `0,00` (label « Score ») couleur cellule via `[Color Score Risque]`
  - `[Niveau Urgence Icon]` (label « Niveau ») — affiche 🔴 / 🟠 / 🟢
- Tri : descendant sur Score
- En-tête : police Segoe UI Bold 12pt blanc, fond `#1A2436`

> 🎓 **METHODE — Couleur cellule par mesure dans une table**
>
> Sur la colonne `[score_risque]` du tableau :
>
> 1. Sélectionner la table → Format → **Cellules** → **Couleur de la police** ou **Couleur d'arrière-plan**
> 2. fx → **Mettre en forme par : Valeur du champ** → choisir la mesure `[Color Score Risque]`
> 3. La cellule prend la couleur retournée par la mesure (rouge / orange / vert) selon le niveau d'urgence de chaque livraison

**6. Répartition urgence (donut)**

- Type : Anneau (Donut)
- Catégorie : `logitrack_risque_scores[niveau_urgence]`
- Valeur : `COUNTROWS(logitrack_risque_scores)`
- Couleurs : Critique rouge `#FF4D6D`, Élevé orange `#FFB547`, Modéré vert mint `#10D9A3`
- Étiquettes : pas affichées dans le donut (légende suffit pour 3 catégories)
- Trou central : 65 %
- Légende : en bas avec puces colorées « ● Critique  ● Élevé  ● Modéré »



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/powerbi/tuto/05_page_alertes_ml_.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Slicers, navigation, finitions

**Navigation horizontale en haut de chaque page** : 5 textes cliquables « Vue Executive · Corridors · Transporteurs · CSAT & Qualité · Alertes ML ». Item actif : texte cyan `#00D4D4` avec soulignement 2px ; items inactifs : texte gris clair `#B0B8C7`.

> 🎓 **METHODE — Navbar horizontale avec état actif**
>
> Power BI ne gère pas l'état actif natif. Astuce :
>
> 1. Sur chaque page, créer 5 boutons **Texte** (un par section), tous avec **action Navigation de page**
> 2. Sur la page courante, le bouton correspondant à la page active n'a PAS d'action de navigation et utilise un style différent (cyan + soulignement)
> 3. Dupliquer la disposition des 5 boutons sur les 5 pages, en variant à chaque fois le bouton actif

**Slicer global** : 1 slicer déroulant `Calendrier[Annee]` (« Tout / 2023 / 2024 ») en haut à droite. Synchroniser sur toutes les pages : Format → **Synchroniser les segments → cocher toutes les pages**.

**Logo « LogiTrack »** : carré cyan `#00D4D4` avec icône camion blanc en haut-gauche, suivi du nom **LogiTrack** blanc 16pt et sous-titre « Supply Chain Analytics » gris clair 11pt.

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** : 10 tables sources + Calendrier + _Mesures, 5 relations actives, 0 LocalDateTable · `Calendrier` marquée comme table de dates · Auto Date/Time désactivé.

**Mesures** : 56 dans `_Mesures` · 8 dossiers numérotés `0.` à `7.` + `_Helpers` · format défini (FCFA, %, jours, ratio).

**Pages** : 5 pages avec sous-titre dynamique · Slicer Année synchronisé · Navbar horizontale avec état actif · Couleurs conformes à la charte Dark + Cyan.

**Performance** : ouverture < 5 s · aucun visuel en erreur.

## 8.2 Pièges fréquents

| Symptôme | Cause | Correction |
|---|---|---|
| `CSAT Moyen` plante en agrégat | Colonne csat en String avec point décimal | `IFERROR(VALUE(SUBSTITUTE([csat], ".", ",")), BLANK())` |
| `RANKX` retourne 1 partout dans une table | Pas de transition de contexte ligne → filtre | Wrapper la mesure dans `CALCULATE([Mesure])` à l'intérieur du RANKX |
| `DATEADD(-1 MONTH)` renvoie blank | `Calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| Bandeau alerte ne change pas de couleur | Power BI ne change pas le fond d'une zone de texte par formule | Utiliser une Carte avec mesure couleur, ou 2 zones superposées avec visibilité conditionnelle |
| Couleur cellule Score Risque uniforme | Mise en forme conditionnelle sur Couleur de fond et non Couleur du champ | fx → Valeur du champ → mesure `[Color Score Risque]` |
| Barres empilées 100 % ne fonctionnent pas | 1 seule mesure utilisée | Glisser 2 mesures complémentaires (Breach Ratio + Conforme Ratio) |
| Combo chart 2 axes Y : 2 lignes invisibles | Axe secondaire pas activé | Format → Axe Y → Plage de valeurs → Activer Axe secondaire |
| Bouton actif sidebar ne change pas selon la page | Power BI ne gère pas l'état actif natif | Dupliquer le bouton avec le style actif sur chaque page |

## 8.3 Storytelling exécutif

Pour présenter au directeur supply chain, suis l'ordre des 5 pages :

1. **Vue Executive** : « 15 000 livraisons sur la période. Taux de breach 39 % (au-dessus de la cible 35 %). CSAT 3,65/5 (en-dessous de 4). 280 K FCFA de pénalités. Tendance breach mensuelle entre 36 et 42 %. »
2. **Corridors** : « Top 5 corridors les plus breachés : Ghana→Mali, Côte d'Ivoire→Ghana, Côte d'Ivoire→Sénégal, Cameroun→Ghana, Mali→Ghana. Corrélation distance vs breach : confirmée (longs corridors → plus de retards). Pénalités max 49,9 K FCFA sur Mali→Ghana. »
3. **Transporteurs** : « 14 actifs, 10 conformes, 4 à surveiller. Retard moyen 6,3 j (en amélioration -0,1). Pénalités moyennes 20,1 K FCFA / transporteur. NordSud Express, Sahel Transport et BurkinaFret en queue de classement. »
4. **CSAT** : « Corrélation inverse confirmée : quand breach monte, CSAT baisse. 41,8 % des livraisons à CSAT ≥ 4 (en hausse +0,3 pts). 920 escalades = 6,1 % du volume. Pires corridors : Mali→Ghana (CSAT 2,87, breach 94 %). »
5. **Alertes ML** : « 25 livraisons en risque critique aujourd'hui — appel dispatcher requis. 2 en alerte élevée. Score moyen 0,83. Top 5 prioritaires : Cameroun→Côte d'Ivoire (Urgent, score 0,91), Cameroun→Ghana (0,90), etc. »
6. **Recommandation** : renforcer les 4 transporteurs « à surveiller » + revoir les contrats SLA sur les 5 corridors les plus breachés (Ghana, Mali, Côte d'Ivoire) → projection : -10 pp de breach et -100K FCFA de pénalités sur 6 mois.

## 8.4 Annexes — Mapping mockup PPTX ↔ pages Power BI

| Slide | Background PNG | Page |
|---|---|---|
| 1 | `bg-01-vue-executive.png` | Vue Executive |
| 2 | `bg-02-corridors.png` | Corridors |
| 3 | `bg-03-transporteurs.png` | Transporteurs |
| 4 | `bg-04-csat.png` | CSAT & Qualité |
| 5 | `bg-05-alertes-ml.png` | Alertes ML |


---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">LogiTrack — Supply Chain Analytics</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>